# Preprocessing Data

In [46]:
import pandas as pd
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
from pathlib import Path

## Load Data

In [47]:
data_dir = Path("raw_data") # This document contains all two raw csv data

pages_path = data_dir / "pages_export.csv"
links_path = data_dir / "links_export.csv"

print(pages_path.exists())
print(links_path.exists())

True
True


In [48]:
# Read first 5 rows because row data is very huge
pages = pd.read_csv(pages_path, nrows=5)
links = pd.read_csv(links_path, nrows=5)
print("pages:")
print(pages)

print("\nlinks:")
print(links)

pages:
    id                           title language  content_length  word_count  \
0    1           The Loner (TV series)       en           11901        1804   
1    2                  She's a Hottie       en            5662         923   
2    3  Point of No Return (TV series)       en            5928         992   
3  610          Underwater exploration       en           70564        9767   
4  612                           Adati       en           15214        2141   

                                          categories  infobox  \
0  ["Articles_with_short_description", "Short_des...      NaN   
1  ["CS1:_unfit_URL", "Articles_with_short_descri...      NaN   
2  ["Articles_lacking_sources_from_November_2012"...      NaN   
3  ["Articles_with_short_description", "Short_des...      NaN   
4  ["Pages_using_gadget_WikiMiniAtlas", "CS1_uses...      NaN   

            created_at                                                url  
0  2025-11-29 21:03:28  https://en.wikipedia.org/wi

In [49]:
print(pages.columns.tolist())
print(links.columns.tolist())

print(pages.dtypes)
print(links.dtypes)

['id', 'title', 'language', 'content_length', 'word_count', 'categories', 'infobox', 'created_at', 'url']
['id', 'source_title', 'target_title', 'language', 'position', 'depth', 'created_at', 'source_url', 'target_url']
id                  int64
title              object
language           object
content_length      int64
word_count          int64
categories         object
infobox           float64
created_at         object
url                object
dtype: object
id               int64
source_title    object
target_title    object
language        object
position         int64
depth            int64
created_at      object
source_url      object
target_url      object
dtype: object


# Node Mapping

## Check the unique of Data

In [50]:
pages = pd.read_csv(
    pages_path,
    usecols=["id", "title", "language", "url"]
)


In [51]:
print(len(pages))

100000


In [52]:
print(pages['language'].value_counts())
print(pages['language'].unique())

en    100000
Name: language, dtype: int64
['en']


In [53]:
print("Unique titles:", pages['title'].nunique())
print("Duplicate titles:", pages['title'].duplicated().sum())
print("Unique URLs:", pages["url"].nunique())
print("Duplicate URLs:", pages["url"].duplicated().sum())

Unique titles: 100000
Duplicate titles: 0
Unique URLs: 100000
Duplicate URLs: 0


The result shows in the pages there are only English pages

In [54]:
links_sample = pd.read_csv(
    links_path,
    usecols=["source_title", "target_title"],
    nrows=100000
)

print(links_sample.head())

            source_title                 target_title
0  The Loner (TV series)  20th Century-Fox Television
1  The Loner (TV series)                    Al Checco
2  The Loner (TV series)          Alan Baxter (actor)
3  The Loner (TV series)            Alexander Courage
4  The Loner (TV series)                Allan Sherman


In [55]:
print("Unique source_titles:", links_sample["source_title"].nunique())
print("Duplicate edges:", links_sample.duplicated(subset=["source_title", "target_title"]).sum())

Unique source_titles: 300
Duplicate edges: 0


## Create Mapping

In [56]:
title_to_idx = pd.Series(
    np.arange(len(pages)),
    index= pages['title']
)

print(title_to_idx.head())

title
The Loner (TV series)             0
She's a Hottie                    1
Point of No Return (TV series)    2
Underwater exploration            3
Adati                             4
dtype: int32


In [57]:
links_sample['source_idx'] = links_sample['source_title'].map(title_to_idx)
links_sample['target_idx'] = links_sample['target_title'].map(title_to_idx)
print(links_sample.head())

            source_title                 target_title  source_idx  target_idx
0  The Loner (TV series)  20th Century-Fox Television           0         NaN
1  The Loner (TV series)                    Al Checco           0         NaN
2  The Loner (TV series)          Alan Baxter (actor)           0         NaN
3  The Loner (TV series)            Alexander Courage           0         NaN
4  The Loner (TV series)                Allan Sherman           0     19619.0


In [58]:
source_match = links_sample['source_idx'].notna().mean()
target_match = links_sample['target_idx'].notna().mean()

both_match = (
    links_sample['source_idx'].notna() & links_sample['target_idx'].notna()
)

print("Source match:", source_match)
print("Target match:", source_match)
print("Both match:", source_match)

Source match: 1.0
Target match: 1.0
Both match: 1.0


# Create Sparse Adjacency Matrix

In [59]:
source = links_sample["source_idx"].to_numpy(dtype=np.int64)
target = links_sample["target_idx"].to_numpy(dtype=np.int64)

f:\anaconda3\envs\nplenv\lib\site-packages\pandas\core\base.py:535: RuntimeWarning: invalid value encountered in cast
  result = np.asarray(self._values, dtype=dtype)


In [60]:
print(links_sample[["source_idx", "target_idx"]].isna().sum())

source_idx        0
target_idx    71912
dtype: int64


In [61]:
links_url_sample = pd.read_csv(
    links_path,
    usecols=["target_url"],
    nrows=100_000
)

url_match = links_url_sample["target_url"].isin(
    pages["url"]
)

print("Target URL match rate:", url_match.mean())

Target URL match rate: 0.28088


In [62]:
check = pd.read_csv(
    links_path,
    usecols=["target_title", "target_url"],
    nrows=100_000
)

title_match = check["target_title"].isin(pages["title"])
url_match = check["target_url"].isin(pages["url"])

print("Different matching results:",
      (title_match != url_match).sum())

Different matching results: 0


In [63]:
unmatched = links_sample[
    links_sample["target_idx"].isna()
]

print(unmatched[
    ["source_title", "target_title"]
].head(10).to_string(index=False))

         source_title                target_title
The Loner (TV series) 20th Century-Fox Television
The Loner (TV series)                   Al Checco
The Loner (TV series)         Alan Baxter (actor)
The Loner (TV series)           Alexander Courage
The Loner (TV series)              Allen H. Miner
The Loner (TV series)               American west
The Loner (TV series)                 Anne Baxter
The Loner (TV series)      Barry Sullivan (actor)
The Loner (TV series)                  Bart Burns
The Loner (TV series)                Beau Bridges


In [64]:
valid_links = links_sample.dropna(
    subset=['source_idx','target_idx']
).copy()

print("Original links:", len(links_sample))
print("Valid links:", len(valid_links))
print("Removed links:", len(links_sample) - len(valid_links))

Original links: 100000
Valid links: 28088
Removed links: 71912


In [65]:
source = valid_links["source_idx"].to_numpy(dtype=np.int64)

target = valid_links["target_idx"].to_numpy(dtype=np.int64)

print(source[:5])
print(target[:5])

[0 0 0 0 0]
[19619 13114 43374  7077 41548]


In [66]:
data = np.ones(len(source), dtype=np.float64)

n = len(pages)

A_csr = sparse.csr_matrix(
    (data, (target, source)),
    shape=(n,n)
)

# why csr: CSR didn't need iterate all zero elements in matrix
# can directly find the non-zero elements in each row
A_csr.sum_duplicates()
A_csr.data[:] = 1.0

print("Matrix shape:", A_csr.shape)
print("Unique edges:", A_csr.nnz) #nnz: Number of Non-zero Entries

Matrix shape: (100000, 100000)
Unique edges: 28088


# Chunk processing

In [67]:
from itertools import islice

total_links = 0
valid_links = 0

A_demo = sparse.csr_matrix((n,n), dtype=np.int32)

reader = pd.read_csv(links_path, usecols=["source_title", "target_title"], chunksize=100000)

for i, chunk in enumerate(islice(reader, 3), start=1):
    source_idx = chunk['source_title'].map(title_to_idx)
    target_idx = chunk['target_title'].map(title_to_idx)

    valid = source_idx.notna() & target_idx.notna()

    source = source_idx[valid].to_numpy(dtype=np.int32)
    target = target_idx[valid].to_numpy(dtype=np.int32)

    data = np.ones(len(source), dtype=np.int32)

    A_chunk = sparse.csr_matrix(
        (data,(target, source)),
        shape=(n,n)
    )

    A_demo = A_demo.maximum(A_chunk)

    total_links += len(chunk)
    valid_links += len(source)

    print(
        f"Chunk{i}:"
        f"read={len(chunk)},"
        f"valid={len(source)},"
        f"graph_edges={A_demo.nnz}"
    )

print("Toal rows processed:",total_links)
print("Valid link records:", valid_links)
print("Unique internal edges:", A_demo.nnz)

Chunk1:read=100000,valid=28088,graph_edges=28088
Chunk2:read=100000,valid=53774,graph_edges=81862
Chunk3:read=100000,valid=53981,graph_edges=135843
Toal rows processed: 300000
Valid link records: 135843
Unique internal edges: 135843


In [68]:
import pandas as pd
import numpy as np

# Step 1: 读取 Pages 文件
pages_check = pd.read_csv(
    pages_path,
    usecols=["title"]
)

# Step 2: 建立网页标题到整数索引的映射
title_to_idx_check = pd.Series(
    np.arange(len(pages_check)),
    index=pages_check["title"]
)

# Step 3: 读取 Links 文件的前 100,000 行
links_check = pd.read_csv(
    links_path,
    usecols=["source_title", "target_title"],
    nrows=100_000
)

# Step 4: 将网页标题映射成索引
source_idx_check = links_check["source_title"].map(title_to_idx_check)
target_idx_check = links_check["target_title"].map(title_to_idx_check)

# Step 5: 判断 Source 和 Target 是否都能匹配
valid_check = source_idx_check.notna() & target_idx_check.notna()

# Step 6: 输出统计结果
print("Total links:", len(links_check))
print("Source unmatched:", source_idx_check.isna().sum())
print("Target unmatched:", target_idx_check.isna().sum())
print("Valid link records:", valid_check.sum())

Total links: 100000
Source unmatched: 0
Target unmatched: 71912
Valid link records: 28088


In [69]:
print("Original mapping size:", len(title_to_idx))
print("Check mapping size:", len(title_to_idx_check))

print(
    "Mapping keys identical:",
    title_to_idx.index.equals(title_to_idx_check.index)
)

print(
    "Mapping values identical:",
    np.array_equal(
        title_to_idx.to_numpy(),
        title_to_idx_check.to_numpy()
    )
)

Original mapping size: 100000
Check mapping size: 100000
Mapping keys identical: True
Mapping values identical: True


In [70]:
# 使用旧映射
source_old = links_check["source_title"].map(title_to_idx)
target_old = links_check["target_title"].map(title_to_idx)

valid_old = source_old.notna() & target_old.notna()

print("Old mapping valid links:", valid_old.sum())

# 使用新映射
print("New mapping valid links:", valid_check.sum())

Old mapping valid links: 28088
New mapping valid links: 28088


In [77]:
# number of nodes
n = len(pages)

total_links = 0
valid_links = 0

# Create Node Mapping
page_index_map = pd.Series(
    np.arange(n, dtype=np.int32),
    index=pages["title"].to_numpy()
)

# check Mapping
assert page_index_map.min() == 0
assert page_index_map.max() == n - 1
assert page_index_map.nunique() == n

# create two list for save valid
source_parts = []
target_parts = []

reader = pd.read_csv(
    links_path,
    usecols = ['source_title','target_title'],
    chunksize=100000
)

for i, chunk in enumerate(reader, start=1):
    
    # Mapping
    source_idx = chunk['source_title'].map(page_index_map)
    target_idx = chunk['target_title'].map(page_index_map)

    # Filtering
    valid = source_idx.notna() & target_idx.notna()

    # Extract valid
    source = source_idx[valid].to_numpy(dtype=np.int32)
    target = target_idx[valid].to_numpy(dtype=np.int32)

    # save Chunk to lists
    if len(source) > 0:

        assert source.min() >= 0
        assert source.max() < n

        assert target.min() >= 0
        assert target.max() < n


        source_parts.append(source)
        target_parts.append(target)

    total_links += len(chunk)
    valid_links += len(source)

    # output every 10 chunks
    if i == 1 or i % 10 == 0:
        print(
            f"Chunk {i}",
            f"total={total_links:,},"
            f"valid={valid_links:,}"
        )

print("Total rows processed:", total_links)
print("Valid link recors:", valid_links)

Chunk 1 total=100,000,valid=28,088
Chunk 10 total=1,000,000,valid=316,357
Chunk 20 total=2,000,000,valid=576,171
Chunk 30 total=3,000,000,valid=841,304
Chunk 40 total=4,000,000,valid=1,090,507
Chunk 50 total=5,000,000,valid=1,349,376
Chunk 60 total=6,000,000,valid=1,823,997
Chunk 70 total=7,000,000,valid=2,269,185
Chunk 80 total=8,000,000,valid=2,709,354
Chunk 90 total=9,000,000,valid=3,172,312
Chunk 100 total=10,000,000,valid=3,629,875
Chunk 110 total=11,000,000,valid=4,091,952
Chunk 120 total=12,000,000,valid=4,540,976
Chunk 130 total=13,000,000,valid=4,982,168
Chunk 140 total=14,000,000,valid=5,427,150
Chunk 150 total=15,000,000,valid=5,888,903
Chunk 160 total=16,000,000,valid=6,339,025
Chunk 170 total=17,000,000,valid=6,788,921
Chunk 180 total=18,000,000,valid=7,001,762
Chunk 190 total=19,000,000,valid=7,146,280
Chunk 200 total=20,000,000,valid=7,288,671
Chunk 210 total=21,000,000,valid=7,432,957
Chunk 220 total=22,000,000,valid=7,578,557
Chunk 230 total=23,000,000,valid=7,725,686


In [80]:
source_all = np.concatenate(source_parts)
target_all = np.concatenate(target_parts)

data = np.ones(len(source_all), dtype=np.int32)

A = sparse.csr_matrix(
    (data, (target_all, source_all)),
    shape=(n,n)
)

A.sum_duplicates()

A.data[:] = 1

A.eliminate_zeros()

print("Matrix shape:", A.shape)
print("Valid link records:", valid_links)
print("Unique internal edges:", A.nnz)
print("Duplicate records:", valid_links - A.nnz)


Matrix shape: (100000, 100000)
Valid link records: 8577044
Unique internal edges: 8577044
Duplicate records: 0


In [81]:
sparse.save_npz("wikipedia_A.npz", A)

In [78]:
print("n =", n)
print("Number of pages =", len(pages))

print("Source minimum =", source_all.min())
print("Source maximum =", source_all.max())

print("Target minimum =", target_all.min())
print("Target maximum =", target_all.max())

print("Mapping size =", len(title_to_idx))
print("Mapping maximum =", title_to_idx.max())

n = 100000
Number of pages = 100000
Source minimum = 100000
Source maximum = 100000
Target minimum = 100000
Target maximum = 100000
Mapping size = 100000
Mapping maximum = 100000


In [82]:
# Calculate Out-degree
out_degree = np.asarray(A.sum(axis=0)).ravel()

# Identify  Dangling Nodes
dangling = (out_degree == 0)

# Display results
print("Number of nodes:", len(out_degree))
print("Total edges:", out_degree.sum())
print("Average out-degree:", out_degree.mean())
print("Maximum out-degree:", out_degree.max())
print("Dangling nodes:", dangling.sum())
print("Dangling ratio:", dangling.mean())

Number of nodes: 100000
Total edges: 8577044
Average out-degree: 85.77044
Maximum out-degree: 1765
Dangling nodes: 204
Dangling ratio: 0.00204


In [83]:
# Inverse out-degree
inv_degree = np.zeros(n, dtype=np.float64)

np.divide(
    1.0,
    out_degree,
    out=inv_degree,
    where=(out_degree != 0)
)

# Construct normalized sparse matrix S
S = A @ sparse.diags(inv_degree, format = 'csr')

# Check matrix properties
print('S shape:', S.shape)
print('S nnz:', S.nnz)

S shape: (100000, 100000)
S nnz: 8577044


In [84]:
# Uniform personalization vector
v = np.full(n, 1.0 / n)

def apply_P(x):
    # Sparse matrix-vector multiplication
    y = S @ x

    # Probability mass currently at dangling nodes
    dangling_mass = x[dangling].sum()

    # Redistribute dangling mass
    y = y + dangling_mass * v

    return y

In [85]:
# Column sums of S
col_sums_S = np.asarray(S.sum(axis=0)).ravel()

# Column sums of implicitly corrected P
col_sums_P = col_sums_S + dangling.astype(np.float64)

print(
    "All columns sum to 1:",
    np.allclose(col_sums_P, 1.0)
)

print(
    "Maximum column-sum error:",
    np.max(np.abs(col_sums_P - 1.0))
)

All columns sum to 1: True
Maximum column-sum error: 2.708944180085382e-14


In [86]:
x = np.full(n, 1.0 / n)

y = apply_P(x)

print("Input probability sum:", x.sum())
print("Output probability sum:", y.sum())
print("Minimum output value:", y.min())

Input probability sum: 1.0
Output probability sum: 0.9999999999999991
Minimum output value: 2.6065722379603404e-08


In [87]:
sparse.save_npz("S.npz", S)

np.savez_compressed(
    "m2_parameters.npz",
    dangling=dangling,
    v=v
)